In [0]:
#connect ADLS to Databricks
spark.conf.set("fs.azure.account.key.adlsgen2spark01.dfs.core.windows.net",
                 "Storageaccount -> Security + Networking -> Access Keys-->Key1valuePaste here ")

In [0]:
%sh

mkdir -p /tmp/brazilian-ecommerce

curl -L -o /tmp/brazilian-ecommerce/brazilian-ecommerce.zip \
https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce

ls -lrt /tmp/brazilian-ecommerce/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42.6M  100 42.6M    0     0  11.6M      0  0:00:03  0:00:03 --:--:-- 15.1M


total 43672
-rw-r--r-- 1 spark-7f8cd1e8-4d6d-40ad-9a34-cb spark-7f8cd1e8-4d6d-40ad-9a34-cb 44717580 May  9 08:04 brazilian-ecommerce.zip


In [0]:
%sh

mkdir -p /tmp/brazilian-ecommerce

unzip /tmp/brazilian-ecommerce/brazilian-ecommerce.zip \
-d /tmp/brazilian-ecommerce/

Archive:  /tmp/brazilian-ecommerce/brazilian-ecommerce.zip
  inflating: /tmp/brazilian-ecommerce/olist_customers_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_geolocation_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_order_items_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_order_payments_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_order_reviews_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_orders_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_products_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_sellers_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/product_category_name_translation.csv  


In [0]:
import os


#specify teh directory containing the files
directory = "/tmp/brazilian-ecommerce/"

#iterate through all the files in the directory
for filename in os.listdir(directory):
  if filename.endswith(".csv"):
    filepath = os.path.join(directory, filename)
    try:
      #Read teh csv file into teh dataframe
      df = spark.read.csv(filepath, header=True, inferSchema= True)

      #Print the filename
      print(f"Processing file: {filename}")

      #print the schema
      print(df.printSchema())

      df.show(10)

      print("-" * 50) #separator between files

    except Exception as e:
      print(f"Error processing file: {filename}: {e}")




Processing file: olist_geolocation_dataset.csv
Error processing file: olist_geolocation_dataset.csv: Public DBFS root is disabled. Access is denied on path: /tmp/brazilian-ecommerce/olist_geolocation_dataset.csv

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsDisabledException
	at com.databricks.backend.daemon.data.client.DisabledDatabricksFileSystem.rejectOperation(DisabledDatabricksFileSystem.scala:34)
	at com.databricks.backend.daemon.data.client.DisabledDatabricksFileSystem.getFileStatus(DisabledDatabricksFileSystem.scala:114)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystemV2.$anonfun$getFileStatus$2(DatabricksFileSystemV2.scala:1227)
	at com.databricks.s3a.S3AExceptionUtils$.convertAWSExceptionToJavaIOException(DatabricksStreamUtils.scala:64)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystemV2.$anonfun$getFileStatus$1(DatabricksFileSystemV2.scala:1224)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging

In [0]:
      ##Access is denied to teh dataset from here , so upload manually the zip file and then unzip and write to adls gen2
#OR 2nd way

#read from githubrul


In [0]:
files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

base_url = "https://raw.githubusercontent.com/garima-gupta13/data-engineering/main/AzureProject-BrazillianEcom/Data/"

for file in files:

    file_url = base_url + file

    print(f"\nReading File: {file}")

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_url)

    print(f"Schema for {file}")
    df.printSchema()

    print(f"Sample Data for {file}")
    df.show(5)

    output_name = file.replace(".csv", "")

    # Writing to ADLS Gen2
    df.write.mode("overwrite").parquet(
        f"abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/{output_name}"
    )

    print(f"{file} written successfully to ADLS Gen2")

    print("-" * 70)


Reading File: olist_customers_dataset.csv
Schema for olist_customers_dataset.csv


---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-8298602242471923>, line 27
     21 df = spark.read.format("csv") \
     22     .option("header", "true") \
     23     .option("inferSchema", "true") \
     24     .load(file_url)
     26 print(f"Schema for {file}")
---> 27 df.printSchema()
     29 print(f"Sample Data for {file}")
     30 df.show(5)

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2008, in DataFrame.printSchema(self, level)
   2006     print(self.schema.treeString(level))
   2007 else:
-> 2008     print(self.schema.treeString())

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1981, in DataFrame.schema(self)
   1978 @property
   1979 def schema(self) -> StructType:
   1980     # self._schema call will cache the schema and serialize it if it is not cached yet.
-> 1981     _schema = self._schema
   1982     if self._cached_schem

In [0]:
import requests
import os

files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

base_url = "https://raw.githubusercontent.com/garima-gupta13/data-engineering/main/AzureProject-BrazillianEcom/Data/"

download_dir = "/tmp/brazilian-ecommerce"

os.makedirs(download_dir, exist_ok=True)

for file in files:

    file_url = base_url + file
    local_path = f"{download_dir}/{file}"

    print(f"\nDownloading: {file}")

    response = requests.get(file_url)

    with open(local_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded to: {local_path}")

    # Read with Spark
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"file:{local_path}")

    print(f"Schema for {file}")
    df.printSchema()

    df.show(5)

    # Write to ADLS
    output_name = file.replace(".csv", "")

    df.write.mode("overwrite").parquet(
        f"abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/{output_name}"
    )

    print(f"{file} written to ADLS successfully")

    print("-" * 70)


Downloading: olist_customers_dataset.csv
Downloaded to: /tmp/brazilian-ecommerce/olist_customers_dataset.csv
Schema for olist_customers_dataset.csv


---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-8298602242471924>, line 43
     37 df = spark.read.format("csv") \
     38     .option("header", "true") \
     39     .option("inferSchema", "true") \
     40     .load(f"file:{local_path}")
     42 print(f"Schema for {file}")
---> 43 df.printSchema()
     45 df.show(5)
     47 # Write to ADLS

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2008, in DataFrame.printSchema(self, level)
   2006     print(self.schema.treeString(level))
   2007 else:
-> 2008     print(self.schema.treeString())

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1981, in DataFrame.schema(self)
   1978 @property
   1979 def schema(self) -> StructType:
   1980     # self._schema call will cache the schema and serialize it if it is not cached yet.
-> 1981     _schema = self._schema
   1982     if self._cached_schema_ser

#Above didnot work as asscess to the tmp is not alloud in free shared ones - so download in the VOLUME & upload to ADLS gen2 as CSV & Parquet

#SAVING as PARQUET

In [0]:
import requests

# All dataset files
files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

# GitHub raw base URL
base_url = "https://raw.githubusercontent.com/garima-gupta13/data-engineering/main/AzureProject-BrazillianEcom/Data/"

# Databricks Volume path
volume_path = "/Volumes/databricks_spark_learning/default/olist-data/"

for file in files:

    print("=" * 80)
    print(f"Processing File: {file}")

    # Full GitHub raw URL
    file_url = base_url + file

    # Save path in Volume
    volume_file_path = volume_path + file

    # ---------------------------------------------------
    # STEP 1 -> Download file from GitHub into Volume
    # ---------------------------------------------------

    print(f"Downloading from GitHub: {file_url}")

    response = requests.get(file_url)

    with open(volume_file_path, "wb") as f:
        f.write(response.content)

    print(f"Saved into Volume: {volume_file_path}")

    # ---------------------------------------------------
    # STEP 2 -> Read file using Spark
    # ---------------------------------------------------

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(volume_file_path)

    # ---------------------------------------------------
    # STEP 3 -> Basic EDA
    # ---------------------------------------------------

    print(f"\nSchema for {file}")
    df.printSchema()

    print(f"\nSample Data for {file}")
    df.show(5)

    print(f"\nTotal Rows in {file}")
    print(df.count())

    print(f"\nTotal Columns in {file}")
    print(len(df.columns))

    print(f"\nColumn Names in {file}")
    print(df.columns)

    # ---------------------------------------------------
    # STEP 4 -> Write to ADLS Gen2
    # ---------------------------------------------------

    output_name = file.replace(".csv", "")

    adls_output_path = f"abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/parquet/{output_name}"

    df.write.mode("overwrite").parquet(adls_output_path)

    print(f"\nWritten successfully to ADLS:")
    print(adls_output_path)

    print("=" * 80)

Processing File: olist_customers_dataset.csv
Saved into Volume: /Volumes/databricks_spark_learning/default/olist-data/olist_customers_dataset.csv

Schema for olist_customers_dataset.csv
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


Sample Data for olist_customers_dataset.csv
+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            S

# Saving as CSV instead of Parquet

In [0]:
files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

volume_path = "/Volumes/databricks_spark_learning/default/olist-data/"

for file in files:

    print("=" * 80)
    print(f"Processing File: {file}")

    # ---------------------------------------------------
    # STEP 1 -> Read CSV from Databricks Volume
    # ---------------------------------------------------

    input_path = volume_path + file

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(input_path)

    print(f"Successfully Read: {file}")

    # ---------------------------------------------------
    # STEP 2 -> Write CSV into ADLS Gen2
    # ---------------------------------------------------

    output_name = file.replace(".csv", "")

    adls_csv_path = f"abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/{output_name}"

    df.write.mode("overwrite") \
        .option("header", "true") \
        .csv(adls_csv_path)

    print(f"Written successfully to ADLS CSV path:")
    print(adls_csv_path)

    print("=" * 80)

Processing File: olist_customers_dataset.csv
Successfully Read: olist_customers_dataset.csv
Written successfully to ADLS CSV path:
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_customers_dataset
Processing File: olist_geolocation_dataset.csv
Successfully Read: olist_geolocation_dataset.csv
Written successfully to ADLS CSV path:
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_geolocation_dataset
Processing File: olist_order_items_dataset.csv
Successfully Read: olist_order_items_dataset.csv
Written successfully to ADLS CSV path:
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_order_items_dataset
Processing File: olist_order_payments_dataset.csv
Successfully Read: olist_order_payments_dataset.csv
Written successfully to ADLS CSV path:
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_order_payments_dataset
Processing File: olist_order_reviews_dataset.csv
Successfully Read: olist_order_reviews_dataset.csv
Written succe